# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

repo_root = Path.cwd()
while repo_root.name != "flyrank-internship" and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "scripts"))

from ml_utils import normalize, percentile_rank

feature_path = repo_root / "data" / "processed" / "refresh_feature_vector.csv"
if not feature_path.exists():
    raise FileNotFoundError(f"Missing processed data at {feature_path}")

frame = pd.read_csv(feature_path)
frame = frame.dropna(subset=["is_declining_label"])


def reason_codes(row: pd.Series) -> list[str]:
    reasons: list[str] = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if str(row["trend_direction"]).lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if row["word_count"] > 0 and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        reasons.append("page_one_decay_risk")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if row["sessions_90d"] >= 30 and (
        (row["engagement_rate"] > 0 and row["engagement_rate"] < 30)
        or (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)
    ):
        reasons.append("low_engagement_visible_page")
    if not reasons:
        reasons.append("general_refresh_review")
    return reasons


def suggested_action(row: pd.Series) -> str:
    reasons = set(str(row["reason_codes"]).split("|"))
    if "thin_visible_page" in reasons:
        return "expand_and_refresh"
    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"
    if "stale_visible_page" in reasons or "declining_with_demand" in reasons:
        return "refresh"
    return "monitor"

frame["visibility_score"] = percentile_rank(np.log1p(frame["impressions_90d"]))
frame["freshness_risk_score"] = percentile_rank(frame["days_since_last_update"])
frame["position_opportunity_score"] = (
    (1 - normalize(frame["avg_position"].clip(lower=1, upper=50)))
    * frame["visibility_score"]
    * (frame["avg_position"] > 0).astype(int)
)
frame["depth_gap_score"] = (1 - percentile_rank(frame["word_count"])) * frame["visibility_score"]

frame["baseline_refresh_score"] = (
    0.40 * frame["visibility_score"]
    + 0.30 * frame["freshness_risk_score"]
    + 0.25 * frame["position_opportunity_score"]
    + 0.05 * frame["depth_gap_score"]
).clip(0, 1)
frame["reason_codes"] = frame.apply(lambda row: "|".join(reason_codes(row)), axis=1)
frame["suggested_action_baseline"] = frame.apply(suggested_action, axis=1)
frame["baseline_rank"] = frame["baseline_refresh_score"].rank(method="first", ascending=False).astype(int)

queue_columns = [
    "baseline_rank",
    "baseline_refresh_score",
    "suggested_action_baseline",
    "reason_codes",
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "trend_direction",
]
action_queue = frame[queue_columns].sort_values("baseline_rank").reset_index(drop=True)

print("Action queue loaded and ranked. Top 5 recommendations:")
action_queue.head(5)

Action queue loaded and ranked. Top 5 recommendations:


,baseline_rank,baseline_refresh_score,suggested_action_baseline,reason_codes,content_id,client_id,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,content_age_days,days_since_last_update,word_count,trend_direction
0,1,0.941189,refresh,declining_with_demand|page_one_decay_risk|low_...,content_9532f197bbc8,client_4e07408562,309192,2689,1098,2.0,0.87,8.01,28.75,445,104,0.0,down
1,2,0.934889,monitor,page_one_decay_risk|low_engagement_visible_page,content_4d1fe5b32dc2,client_19581e27de,97999,512,549,2.5,0.52,7.47,13.15,329,104,0.0,stable
2,3,0.934080,monitor,page_one_decay_risk|low_engagement_visible_page,content_07f2e7a6f38a,client_19581e27de,101078,856,780,2.7,0.85,2.05,4.60,313,104,0.0,stable
3,4,0.933606,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,content_e5ae436f9a16,client_4e07408562,117741,533,522,3.0,0.45,7.09,12.60,421,104,0.0,stable
4,5,0.933559,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,content_3430a8b94511,client_19581e27de,152617,440,534,3.3,0.29,6.18,11.04,329,104,0.0,stable


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [3]:
intended_use = (
    "Use this playbook as a ranked refresh queue for page-level content with measurable impressions,"
    " sessions, and position history. It is designed to support editorial review by highlighting pages"
    " with high visibility risk, stale content, and clear opportunities in search performance."
)
limits = (
    "Do not use this queue as a final publishing decision or for pages with no traffic history,"
    " brand safety concerns, or editorial strategy needs that are not captured in the available signals."
)

action_counts = (
    action_queue["suggested_action_baseline"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="count")
)

print(intended_use)
print("\n", limits)
print("\nAction distribution: ")
action_counts

Use this playbook as a ranked refresh queue for page-level content with measurable impressions, sessions, and position history. It is designed to support editorial review by highlighting pages with high visibility risk, stale content, and clear opportunities in search performance.

 Do not use this queue as a final publishing decision or for pages with no traffic history, brand safety concerns, or editorial strategy needs that are not captured in the available signals.

Action distribution: 


,action,count
0,monitor,13173
1,refresh_and_review_ctr,9741
2,refresh,7004
3,expand_and_refresh,82


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [4]:
human_review_checks = [
    "Confirm the page content is still on-strategy for the client before any rewrite or refresh.",
    "Verify that high-visibility pages are not under active editorial campaigns or temporary promotions.",
    "Check whether performance issues are caused by tracking, page structure, or metadata rather than content quality.",
    "Review any pages labeled for expansion to ensure the update preserves the user intent and does not add low-value filler.",
    "Never automate deletion, redirect, or site architecture changes without a product and editorial sign-off.",
]
no_go_list = [
    "Do not refresh pages without traffic data or with fewer than two weeks of stable impressions.",
    "Do not use this queue to auto-publish new pages or to bypass editorial triage.",
    "Do not change pages with brand, legal, or sensitive compliance signals without human approval.",
    "Do not treat this output as a content quality score when the page has no documented search intent.",
]

review_examples = action_queue[action_queue["suggested_action_baseline"] != "monitor"].head(10)

print("Human review checklist:")
for item in human_review_checks:
    print(f"- {item}")

print("\nNo-go list:")
for item in no_go_list:
    print(f"- {item}")

print("\nSample recommended refresh pages for review:")
review_examples

Human review checklist:
- Confirm the page content is still on-strategy for the client before any rewrite or refresh.
- Verify that high-visibility pages are not under active editorial campaigns or temporary promotions.
- Check whether performance issues are caused by tracking, page structure, or metadata rather than content quality.
- Review any pages labeled for expansion to ensure the update preserves the user intent and does not add low-value filler.
- Never automate deletion, redirect, or site architecture changes without a product and editorial sign-off.

No-go list:
- Do not refresh pages without traffic data or with fewer than two weeks of stable impressions.
- Do not use this queue to auto-publish new pages or to bypass editorial triage.
- Do not change pages with brand, legal, or sensitive compliance signals without human approval.
- Do not treat this output as a content quality score when the page has no documented search intent.

Sample recommended refresh pages for review:

,baseline_rank,baseline_refresh_score,suggested_action_baseline,reason_codes,content_id,client_id,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,content_age_days,days_since_last_update,word_count,trend_direction
0,1,0.941189,refresh,declining_with_demand|page_one_decay_risk|low_...,content_9532f197bbc8,client_4e07408562,309192,2689,1098,2.0,0.87,8.01,28.75,445,104,0.0,down
3,4,0.933606,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,content_e5ae436f9a16,client_4e07408562,117741,533,522,3.0,0.45,7.09,12.60,421,104,0.0,stable
4,5,0.933559,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,content_3430a8b94511,client_19581e27de,152617,440,534,3.3,0.29,6.18,11.04,329,104,0.0,stable
5,6,0.933263,refresh_and_review_ctr,declining_with_demand|page_one_decay_risk|low_...,content_cbd93118300b,client_19581e27de,145292,662,535,3.3,0.46,1.87,5.38,313,104,0.0,down
8,9,0.931363,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,content_79b25654070a,client_19581e27de,148737,711,619,3.7,0.48,2.26,3.46,257,104,0.0,stable
10,11,0.931033,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,content_8dacab06e291,client_19581e27de,127907,439,381,3.6,0.34,0.26,3.33,313,104,0.0,stable
11,12,0.930401,refresh_and_review_ctr,declining_with_demand|page_one_decay_risk|low_...,content_01908772c6db,client_19581e27de,187893,845,782,4.0,0.45,1.53,3.67,313,104,0.0,down
12,13,0.930217,refresh_and_review_ctr,declining_with_demand|page_one_decay_risk|low_...,content_5fe46e04994d,client_4e07408562,517715,741,520,4.2,0.14,4.23,26.90,537,104,0.0,down
13,14,0.930125,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,content_6f81ccd92b64,client_19581e27de,73675,138,136,2.9,0.19,2.94,3.16,257,104,0.0,stable
14,15,0.930118,refresh_and_review_ctr,declining_with_demand|page_one_decay_risk|low_...,content_c1350d507c68,client_19581e27de,142505,417,398,3.9,0.29,1.01,1.85,313,104,0.0,down


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [5]:
monitoring_triggers = [
    {
        "trigger": "Top-50 queue score drops",
        "condition": "median baseline_refresh_score in top 50 falls below 0.60",
        "why": "A weaker top queue suggests the scoring formula is selecting lower-confidence refresh candidates than expected.",
    },
    {
        "trigger": "Action mix drift",
        "condition": "more than 70% of the queue is 'monitor'",
        "why": "The score may be too conservative, and the queue is not surfacing enough clear refresh opportunities.",
    },
    {
        "trigger": "Visibility signal shift",
        "condition": "median impressions_90d or avg_position changes by more than 20% from the last cycle",
        "why": "Search demand or position patterns are changing, so the current ranking weights may be stale.",
    },
    {
        "trigger": "Editorial review feedback",
        "condition": "editors report more than 20% of recommended pages are not actionable",
        "why": "Human validation is the best check for whether the queue is aligned with current content goals.",
    },
]

queue_summary = action_queue["baseline_refresh_score"].describe(percentiles=[0.25, 0.5, 0.75]).to_frame().T
queue_summary.columns = [str(col) for col in queue_summary.columns]

action_mix = action_counts.copy()
action_mix["pct"] = (action_mix["count"] / len(action_queue)).round(3)

print("Monitoring triggers:")
for trigger in monitoring_triggers:
    print(f"- {trigger['trigger']}: {trigger['condition']} -> {trigger['why']}")

print("\nBaseline score summary:")
queue_summary

print("\nAction mix:")
action_mix

Monitoring triggers:
- Top-50 queue score drops: median baseline_refresh_score in top 50 falls below 0.60 -> A weaker top queue suggests the scoring formula is selecting lower-confidence refresh candidates than expected.
- Action mix drift: more than 70% of the queue is 'monitor' -> The score may be too conservative, and the queue is not surfacing enough clear refresh opportunities.
- Visibility signal shift: median impressions_90d or avg_position changes by more than 20% from the last cycle -> Search demand or position patterns are changing, so the current ranking weights may be stale.
- Editorial review feedback: editors report more than 20% of recommended pages are not actionable -> Human validation is the best check for whether the queue is aligned with current content goals.

Baseline score summary:

Action mix:


,action,count,pct
0,monitor,13173,0.439
1,refresh_and_review_ctr,9741,0.325
2,refresh,7004,0.233
3,expand_and_refresh,82,0.003


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
output_dir = repo_root / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

queue_output_path = output_dir / "action_playbook_queue.csv"
action_counts_output_path = output_dir / "action_playbook_action_counts.csv"
summary_output_path = output_dir / "action_playbook_summary.json"

export_columns = [
    "baseline_rank",
    "baseline_refresh_score",
    "suggested_action_baseline",
    "reason_codes",
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "trend_direction",
]
action_queue[export_columns].to_csv(queue_output_path, index=False)
action_counts.to_csv(action_counts_output_path, index=False)

summary_payload = {
    "rows": int(len(action_queue)),
    "top_score": float(action_queue["baseline_refresh_score"].max()),
    "median_score": float(action_queue["baseline_refresh_score"].median()),
    "monitor_share": float(action_mix[action_mix["action"] == "monitor"]["pct"].squeeze()) if "monitor" in action_mix["action"].values else 0.0,
    "action_counts": action_counts.set_index("action")["count"].to_dict(),
}

import json
summary_output_path.write_text(json.dumps(summary_payload, indent=2))
print(f"Wrote {queue_output_path}")
print(f"Wrote {action_counts_output_path}")
print(f"Wrote {summary_output_path}")

Wrote /home/otto/Documents/projects/flyrank-internship/work/outputs/action_playbook_queue.csv
Wrote /home/otto/Documents/projects/flyrank-internship/work/outputs/action_playbook_action_counts.csv
Wrote /home/otto/Documents/projects/flyrank-internship/work/outputs/action_playbook_summary.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.